# Thành viên 1 — Kaggle `GPU T4 x 2` (32 GB VRAM): Track Tiếng Việt (`ViSoBERT-ViHSD + mDeBERTa Stance Verbalizer -> ViSoBERT CE + SpanPool -> SBi-FAGAT`)

**Cấu hình bắt buộc trên Kaggle trước khi chạy:**
1. Bật **Accelerator: GPU T4 x 2** (tổng 32 GB VRAM).
2. Bật **Internet: ON** để kéo repo GitHub, dữ liệu Arctic Shift và trọng số Hugging Face.
3. Bật **Persistence: Files only** (khuyên dùng) để giữ checkpoint nếu ngắt phiên.

Bạn có 2 chế độ chọn nhanh:
- **`member1_vn`** *(Khuyên dùng để chia 50/50 với Bạn 2)*: Chạy trọn gói Tiếng Việt (thu thập 1.500 cây Voz/VietNam -> tinh chỉnh `uitnlp/visobert` trên `UIT-ViHSD` + chấm `mDeBERTa-v3-xnli` Stance Verbalizer -> chưng cất Cross-Encoder 2D `uitnlp/visobert` -> trích xuất `.pt` *Ancestor SpanPool* -> 5-seed công bằng `SBi-FAGAT` `hidden_dim=64` + xuất file chấm 300 câu).
- **`model_nlp_only`**: Nếu muốn 1 người chuyên train toàn bộ **Mô hình NLP (`ViSoBERT-ViHSD + mDeBERTa Verbalizer + Cross-Encoder 2D`)** xuất sẵn `.pt` rồi gửi sang cho Bạn 2 train GNN.

In [ ]:
# Bước 1: Kiểm tra GPU NVIDIA Tesla T4
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available(), "| Num GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
# Bước 2: Clone hoặc cập nhật mã nguồn mới nhất từ GitHub & cài thư viện
%cd /kaggle/working
!if [ ! -d "gnn-nlp-propagation" ]; then git clone https://github.com/hieu-kiien/gnn-nlp-propagation.git; fi
%cd /kaggle/working/gnn-nlp-propagation
!git pull origin main
!pip install -q -r requirements.txt
!pytest -q

In [ ]:
# Khởi tạo muối băm SHA-256 bền vững (nếu chưa đặt SBIF_HASH_SALT) để giả danh hóa đồng nhất kể cả khi resume
import os, pathlib, secrets
salt_file = pathlib.Path("data/interim/.author_hash_salt.secret")
if not os.environ.get("SBIF_HASH_SALT") and not salt_file.exists():
    salt_file.parent.mkdir(parents=True, exist_ok=True)
    salt_file.write_text(secrets.token_hex(16), encoding="utf-8")
    print("Created persistent salt at", salt_file)

# (Tùy chọn) Đăng nhập Hugging Face nếu repo model yêu cầu token đọc
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# Bước 3: Chạy thử kiểm tra nhanh hệ thống (chạy trong <10 giây để chắc chắn 0 lỗi môi trường)
!python -m scripts.kaggle_parallel_runner --role member1_vn --smoke-test
!python -m scripts.predict_live_thread --demo-offline

In [ ]:
# Bước 4: CHẠY CHÍNH THỨC TRÊN KAGGLE
# Đổi ROLE = "model_nlp_only" nếu bạn chỉ muốn chuyên trách xuất bộ NLP Tiếng Việt (.pt) cho bạn còn lại
ROLE = "member1_vn"  # hoặc "model_nlp_only"
VN_TARGET_COUNT = 1500

!python -m scripts.kaggle_parallel_runner \
    --role {ROLE} \
    --lang vi \
    --vn-target-count {VN_TARGET_COUNT} \
    --before-epoch 1763164800 \
    --hidden-dim 64 \
    --epochs-nlp 3 \
    --epochs-gnn 100

In [ ]:
# Bước 5: Kiểm tra thử trực tiếp 1 bài Reddit thật trên mạng bằng checkpoint vừa train
# Thay --reddit-url bằng link bài tranh luận mới bất kỳ trên r/vozforums hoặc r/VietNam
!python -m scripts.predict_live_thread \
    --demo-offline \
    --cross-encoder-ckpt checkpoints/dual_ce_vn \
    --gnn-ckpt checkpoints/best_sbifagat_vn.pt

## Xuất kết quả & bảo toàn đáp án chấm mù
Sau khi ô trên chạy xong, tải đồng thời **2 gói nén** trong `/kaggle/working/gnn-nlp-propagation/` trước khi tắt máy ảo Kaggle:
1. `sbifagat_member1_vn_bundle.zip` *(hoặc `sbifagat_model_nlp_only_bundle.zip`)* — chứa kết quả chính `paper/results_main_vn.json`, `paper/results_ablation_vn.json`, `paper/tables_*_vn.tex`, `paper/figures/vn/*.pdf`, `checkpoints/best_sbifagat_vn.pt` và phiếu chấm mù `data/interim/human_audit_vi_300.csv` (không kèm đáp án máy).
2. `sbifagat_member1_vn_sealed_machine_keys.zip` — gói niêm phong riêng chứa `data/interim/human_audit_vi_300_machine_key.csv` (chỉ giải nén đối chiếu `machine_vs_gold` sau khi 2 bạn chấm xong phiếu mù).

*Lưu ý:* Ở chế độ mặc định công bằng không chạy `auto_loop` (nếu muốn xuất thêm chẩn đoán Validation `outputs/auto_loop_vn/{loop_diagnostics,worst_val_cases_for_review}.json`, thêm cờ `--run-auto-loop` ở Bước 4).